In [9]:
# TO DO: manually edit file name and file type

FILE_NAME = "Ethanol_Biorefineries.csv"
MIME_TYPE = "text/csv"

In [20]:
# TO DO: just run this code snippet. It'll output column names

import os
import pandas as pd
from ca_biositing.pipeline.utils.gdrive_to_pandas import gdrive_to_df
CREDENTIALS_PATH = "../../../../../../resources/prefect/credentials.json"
DATASET_FOLDER = "../temp_external_datasets/"
print(f"Extracting raw data from '{FILE_NAME}'...")

credentials_path = CREDENTIALS_PATH
dataset_folder = DATASET_FOLDER

raw_df = gdrive_to_df(FILE_NAME, MIME_TYPE, credentials_path, dataset_folder)

if raw_df is None:
    print("Failed to extract data. Aborting.")

print("Successfully extracted raw data.")
print(raw_df.columns)

NameError: name 'FILE_NAME' is not defined

In [4]:
# TO DO: [type later]

from ca_biositing.pipeline.utils.geo_utils import parse_addresses

address_df, geoid_df = parse_addresses(
        raw_df,
        merge_columns=["ADDRESS", "CITY", "STATE"],
        lat=None,
        long=None
    )

added_address_df = pd.concat([address_df, geoid_df], axis=1)

KeyError: 'status'

In [5]:
added_address_df

NameError: name 'added_address_df' is not defined

# Geocoder Module

In [12]:
# only run once
import gspread
from ca_biositing.pipeline.etl.extract.factory import create_extractor

GSHEET_NAME = "test sheet"
WORKSHEET_NAME = "test sheet"

extract = create_extractor(GSHEET_NAME, WORKSHEET_NAME)

# API: Get all values? (change later)
# Mask rows labeled pending and geocode for each of them
# Deal with duplicates
# Update status
# Deal with duplicates?
# Load to locationaddress and get id
# Update ID
# API: Update values


In [24]:
df = extract("../../../../../../resources/prefect/")

06:59:39.439 | INFO    | Task run 'extract_test sheet' - Extracting raw data from 'test sheet' in 'test sheet'...

DEBUG: gsheet_to_df called for test sheet / test sheet
DEBUG: Authenticating with ../../../../../../resources/prefect/credentials.json
DEBUG: Opening spreadsheet test sheet
DEBUG: Opening worksheet by name: test sheet
DEBUG: Fetching all values from worksheet
DEBUG: Successfully fetched 8 rows


06:59:40.681 | INFO    | Task run 'extract_test sheet' - Successfully extracted raw data from test sheet.

/workspaces/ca-biositing/.pixi/envs/default/lib/python3.13/logging/__init__.py:1946: UserWarning: Logger 'prefect.task_runs' attempted to send logs to the API without a flow run id. The API log handler can only send logs within flow run contexts unless the flow run id is manually provided. Set PREFECT_LOGGING_TO_API_WHEN_MISSING_FLOW=ignore to suppress this warning.
  self.logger.log(level, msg, *args, **kwargs)


06:59:40.685 | INFO    | Task run 'extract_test sheet' - Finished in state Completed()

In [25]:
df

,name,address,city,county,zip,state,latitude,longitude,status,closest_address_line_1,...,closest_postal_code,closest_latitude,closest_longitude,closest_geoid,closest_state_name,closest_state_fips,closest_county_name,closest_county_fips,is_na,address_id
0,4 CORNER GROWERS LLC,"9051 AGUAS FRIAS RD, CHICO, 95928, CA, CHICO, ...",CHICO,Butte,95928,CA,39.6333364,-121.8654062,true,9051 Aguas Frias Road,...,95928-9522,39.6333364,-121.8654062,06007,CA,06,Butte County,007,FALSE,1
1,A & R CUNHA FARMS INC,"5069 FRENCH PEOPLE, MEEP MORP, , , MEEP MORP, ...",MEEP MORP,beep,,,37.8737728,-122.2079590,false,100 California Shakespeare Theater Way Siesta ...,...,94563,37.873279,-122.2058206,06013,CA,06,Contra Costa County,013,FALSE,2
2,ADRIAN RANCH,"26554 E. RIVER RD, ESCALON, 95320, CA, ESCALON...",ESCALON,San Joaquin,95320,CA,37.7573830,-120.9821665,true,26554 East River Road,...,95320-9652,37.757383,-120.9821665,06077,CA,06,San Joaquin County,077,FALSE,3
3,ALMOND TREE HULLING CO,"23175 RD 16, CHOWCHILLA, 93610, CA, CHOWCHILLA...",CHOWCHILLA,Madera,93610,CA,37.0861803,-120.2565377,true,23175 Road 16,...,93610-8704,37.0861803,-120.2565377,06039,CA,06,Madera County,039,FALSE,4
4,ALMOND ORCHARDS INC,"9409 TROXEL RD, CHICO, 95938, CA, CHICO, 95938...",CHICO,Butte,95938,CA,39.6478491,-121.8435109,true,9409 Troxel Road,...,95928,39.6478491,-121.8435109,06007,CA,06,Butte County,007,FALSE,1
5,ANTONOWICH HULLING,"2656 HOUSE AVE, DURHAM, 95938, CA, DURHAM, 959...",DURHAM,Butte,95938,CA,39.6251999,-121.8117102,true,2656 House Avenue,...,95938-9763,39.6251999,-121.8117102,06007,CA,06,Butte County,007,FALSE,1
6,ASSOCIATED HULLING AND SHELLING LP,"4506 SAYLOR ROAD, DENAIR, 95316, CA, DENAIR, 9...",DENAIR,Stanislaus,95316,CA,37.5724303,-120.8151822,true,4506 Saylor Road,...,95316-9534,37.5724303,-120.8151822,06099,CA,06,Stanislaus County,099,FALSE,5


In [26]:
from ca_biositing.pipeline.utils.geo_utils import parse_addresses
from dotenv import load_dotenv
import pandas as pd

df = df.astype({'latitude': float, 'longitude': float})

address_df, geoid_df = parse_addresses(
        df,
        merge_columns=["address", "city", "zip", "state"],
        lat="latitude",
        long="longitude",
    )

added_address_df = pd.concat([address_df, geoid_df], axis=1)
added_address_df

,status,closest_address_line_1,closest_address_line_2,closest_city,closest_county,closest_state,closest_postal_code,closest_latitude,closest_longitude,closest_geoid,closest_state_name,closest_state_fips,closest_county_name,closest_county_fips


In [ ]:
replace_columns = added_address_df.columns
df.loc[df["status"] == "pending", replace_columns] = added_address_df

In [10]:
found_addresses = df[df["status"] == "true"]

In [17]:


print("Bridging County (Place) to LocationAddress...")
from sqlmodel import Session, select, create_engine
from ca_biositing.pipeline.utils.engine import _get_database_url, get_engine
from ca_biositing.datamodels.models import LocationAddress, Place
from ca_biositing.datamodels.config import settings

db_url = _get_database_url()
print(db_url)
db_url = db_url.replace("@db:5432", f"@localhost:{settings.POSTGRES_PORT}")
db_url = db_url.replace("@db:", f"@localhost:{settings.POSTGRES_PORT}")
engine = create_engine(
        db_url,
        pool_size=5,
        max_overflow=0,
        pool_pre_ping=True,
        connect_args={"connect_timeout": 10}
)

with Session(engine) as session:
    place_to_address_map = {}

    for index, row in found_addresses.iterrows():
        geoid = row.get("closest_geoid")
        if geoid is not None:
            stmt1 = select(Place).where(Place.geoid == geoid)
            place = session.exec(stmt1).first()

            stmt2 = select(LocationAddress).where(
                LocationAddress.geography_id == geoid
            )
            address = session.exec(stmt2).first()

            if not place:
                place = Place(
                    geoid=geoid,
                    state_name=row.get("closest_state_name"),
                    state_fips=row.get("closest_state_fips"),
                    county_name=row.get("closest_county_name"),
                    county_fips=row.get("closest_county_fips"),
                )
                session.add(place)
                session.flush()

            if not address:
                address = LocationAddress(
                    geography_id=geoid,
                    address_line1=row.get("closest_address_line_1"),
                    address_line2=row.get("closest_address_line_2"),
                    city=row.get("closest_city"),
                    zip=row.get("closest_postal_code"),
                    lat=row.get("closest_latitude"),
                    lon=row.get("closest_longitude"),
                    is_anonymous=False,
                )
                session.add(address)
                session.flush()

            place_to_address_map[geoid] = address.id

    session.commit()
    found_addresses["address_id"] = found_addresses["closest_geoid"].map(
        place_to_address_map
    )
    print(
        f"Mapped {len(place_to_address_map)} counties to LocationAddresses"
    )

Bridging County (Place) to LocationAddress...
postgresql+psycopg2://biocirv_user:biocirv_dev_password@db:5432/biocirv_db
Mapped 5 counties to LocationAddresses


In [27]:
df['address_id'] = 0
df.loc[df["status"] == "true", "address_id"] = found_addresses["address_id"]
df

,name,address,city,county,zip,state,latitude,longitude,status,closest_address_line_1,...,closest_postal_code,closest_latitude,closest_longitude,closest_geoid,closest_state_name,closest_state_fips,closest_county_name,closest_county_fips,is_na,address_id
0,4 CORNER GROWERS LLC,"9051 AGUAS FRIAS RD, CHICO, 95928, CA, CHICO, ...",CHICO,Butte,95928,CA,39.633336,-121.865406,true,9051 Aguas Frias Road,...,95928-9522,39.6333364,-121.8654062,06007,CA,06,Butte County,007,False,1
1,A & R CUNHA FARMS INC,"5069 FRENCH PEOPLE, MEEP MORP, , , MEEP MORP, ...",MEEP MORP,beep,,,37.873773,-122.207959,false,100 California Shakespeare Theater Way Siesta ...,...,94563,37.873279,-122.2058206,06013,CA,06,Contra Costa County,013,False,0
2,ADRIAN RANCH,"26554 E. RIVER RD, ESCALON, 95320, CA, ESCALON...",ESCALON,San Joaquin,95320,CA,37.757383,-120.982167,true,26554 East River Road,...,95320-9652,37.757383,-120.9821665,06077,CA,06,San Joaquin County,077,False,3
3,ALMOND TREE HULLING CO,"23175 RD 16, CHOWCHILLA, 93610, CA, CHOWCHILLA...",CHOWCHILLA,Madera,93610,CA,37.086180,-120.256538,true,23175 Road 16,...,93610-8704,37.0861803,-120.2565377,06039,CA,06,Madera County,039,False,4
4,ALMOND ORCHARDS INC,"9409 TROXEL RD, CHICO, 95938, CA, CHICO, 95938...",CHICO,Butte,95938,CA,39.647849,-121.843511,true,9409 Troxel Road,...,95928,39.6478491,-121.8435109,06007,CA,06,Butte County,007,False,1
5,ANTONOWICH HULLING,"2656 HOUSE AVE, DURHAM, 95938, CA, DURHAM, 959...",DURHAM,Butte,95938,CA,39.625200,-121.811710,true,2656 House Avenue,...,95938-9763,39.6251999,-121.8117102,06007,CA,06,Butte County,007,False,1
6,ASSOCIATED HULLING AND SHELLING LP,"4506 SAYLOR ROAD, DENAIR, 95316, CA, DENAIR, 9...",DENAIR,Stanislaus,95316,CA,37.572430,-120.815182,true,4506 Saylor Road,...,95316-9534,37.5724303,-120.8151822,06099,CA,06,Stanislaus County,099,False,5


In [23]:
gc = gspread.service_account(filename=CREDENTIALS_PATH)
sh = gc.open('test sheet')
worksheet = sh.worksheet("test sheet")
worksheet.update([df.columns.values.tolist()] + df.values.tolist())

{'spreadsheetId': '1uIReRKplM5cGLy-5FCOikhUuhKC6ZrLH2tGihAxo1is',
 'updatedRange': "'test sheet'!A1:X8",
 'updatedRows': 8,
 'updatedColumns': 24,
 'updatedCells': 192}